[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/MDP_introduction_frozenlake_slippery.ipynb)

# The Slippery Frozen Lake: a full Markov Decision Process, step by step

The companion notebook `MDP_introduction_frozenlake.ipynb` walks through Frozen Lake on
**firm ice** (`is_slippery=False`), where the world is *deterministic*: the action you
choose is the move you make. This notebook does the same walkthrough on **slippery ice**
(`is_slippery=True`), where the world is *stochastic*: you intend to go one way but the
ice can send you sideways.

That single change &mdash; deterministic &rarr; stochastic &mdash; is the entire reason
Reinforcement Learning needs *probability*. Our goal here is to make the slippery MDP
completely explicit: we will read the exact transition probabilities out of the
environment, understand the reward and terminal structure, and then **solve the MDP by
planning** (value iteration) to find the optimal policy. Along the way we will see why
the optimal slippery policy looks, at first glance, completely wrong.

## What is a Markov Decision Process?

An MDP is the formal description of the agent's world. It is a 5-tuple $(S, A, P, R, \gamma)$:

| Symbol | Name | On Frozen Lake |
|--------|------|----------------|
| $S$ | set of **states** | the 16 grid cells (0&ndash;15) |
| $A$ | set of **actions** | 4 moves: Left, Down, Right, Up |
| $P(s' \mid s, a)$ | **transition** probabilities | *the part that changes when the ice is slippery* |
| $R(s, a, s')$ | **reward** function | +1 for reaching the goal, else 0 |
| $\gamma$ | **discount** factor | how much we value future reward (we use 0.99) |

On firm ice, $P(s' \mid s, a)$ is trivial: probability 1 on the intended cell. On slippery
ice it spreads probability across several cells. **Everything interesting in this notebook
lives in $P$.** The "Markov" in MDP means $P$ depends only on the *current* state and
action &mdash; not on how the agent got there.

In [ ]:
# Setup: on Google Colab, install Gymnasium and grab the shared plotting helpers.
import sys, os
if 'google.colab' in sys.modules:
    !pip install -qq gymnasium==1.3.0 pygame seaborn
    if not os.path.exists('utils_frozenlake.py'):
        !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils_frozenlake.py

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from utils_frozenlake import plot_values, plot_policy, test_agent, evaluate_policy

## Build the slippery environment

The only difference from the firm-ice notebook is `is_slippery=True`. We keep the standard
4&times;4 map so we can compare results one-to-one with the deterministic version.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)   # is_slippery=True -> stochastic ice

n_states  = env.observation_space.n   # 16 cells
n_actions = env.action_space.n        # 4 moves
ACTIONS = ["Left", "Down", "Right", "Up"]   # index 0..3, the order Gymnasium uses

print(f"{n_states} states, {n_actions} actions")
print("The 4x4 map (S=start, F=frozen, H=hole, G=goal):")
for row in env.unwrapped.desc:                       # desc is the raw map as bytes
    print("   ", b" ".join(row).decode())

## States and actions (unchanged by slipperiness)

The state is just *which cell the agent is on*, numbered 0 (top-left) to 15 (bottom-right),
counting left-to-right, top-to-bottom. The four actions are the four moves. Slipperiness
does **not** change what the states or actions *are* &mdash; it only changes what *happens*
when you take an action. So this part is identical to the firm-ice notebook.

In [ ]:
state, info = env.reset()          # reset() drops the agent on the start cell (state 0)
print("Start state:", state)

# Map each cell number to its (row, col) so we can talk about positions.
def rc(s):
    return divmod(s, 4)            # divmod(s, 4) -> (row, col) on a 4-wide grid
print("State 6 is at (row, col) =", rc(6))
print("A random example action:", env.action_space.sample(), "->", ACTIONS[env.action_space.sample()])

## The heart of it: the transition model $P(s' \mid s, a)$

On slippery ice, choosing an action does **not** guarantee that move. Frozen Lake uses this
rule: the intended direction happens with probability **1/3**, and the two directions
**perpendicular** to it happen with probability **1/3** each. You never move in the exact
opposite of your intended direction. If a resulting move would go off the grid, the agent
just stays in place.

So "go Right" really means: *1/3 Right, 1/3 Up, 1/3 Down.* That is the whole source of the
difficulty &mdash; and Gymnasium hands us these probabilities exactly, in
`env.unwrapped.P`.

`env.unwrapped.P[s][a]` is a list of outcomes, each a tuple
`(probability, next_state, reward, terminated)`.

In [ ]:
P = env.unwrapped.P     # P[s][a] -> list of (probability, next_state, reward, terminated)

# Read the model for one concrete choice: from the START cell (0), intend to go Down (action 1).
print("From state 0, action 'Down' -> possible outcomes:")
for prob, next_state, reward, terminated in P[0][1]:
    # Each line is one way the ice can send us, with its probability.
    print(f"   p={prob:.3f}  ->  state {next_state} at {divmod(next_state,4)}"
          f"   reward={reward}  terminated={terminated}")

Notice three outcomes of probability 1/3 each, and none of them is guaranteed to be the
cell directly below the start. That is the slip. Let's write a small helper so we can
inspect the slip distribution for *any* state and action, and reuse it below.

In [ ]:
def show_transition(s, a):
    """Pretty-print the probability distribution over next states for taking action a in state s."""
    print(f"State {s} {divmod(s,4)}, action '{ACTIONS[a]}':")
    for prob, ns, r, term in P[s][a]:
        tag = "  <-- GOAL" if r == 1 else ("  <-- hole/terminal" if term else "")
        print(f"   p={prob:.3f} -> state {ns} {divmod(ns,4)}{tag}")

# State 6 is a middle cell with holes nearby, so its slips are especially telling.
show_transition(6, 2)   # intend Right...

## Firm ice vs slippery ice, side by side

The cleanest way to *feel* the difference is to read the same `(state, action)` out of both
worlds. On firm ice the distribution is a single certain outcome; on slippery ice it is
spread over three cells.

In [ ]:
env_firm = gym.make("FrozenLake-v1", is_slippery=False)   # deterministic twin
P_firm = env_firm.unwrapped.P

def compare(s, a):
    print(f"=== state {s} {divmod(s,4)}, action '{ACTIONS[a]}' ===")
    print(" firm ice (deterministic):")
    for prob, ns, r, term in P_firm[s][a]:
        print(f"    p={prob:.3f} -> state {ns}")
    print(" slippery ice (stochastic):")
    for prob, ns, r, term in P[s][a]:
        print(f"    p={prob:.3f} -> state {ns}")

compare(6, 2)   # same cell, same intended move, two different worlds

## Rewards and terminal states

The reward structure is unchanged by slipperiness: **+1 only on the transition into the goal
(state 15)**, and **0 everywhere else**. Holes and the goal are *terminal*: once you land on
one the episode ends (`terminated=True`). This is a **sparse-reward** problem &mdash; the
agent gets exactly one non-zero signal per successful episode, which is what makes credit
assignment hard.

We can confirm this by scanning the whole model for any transition that pays a reward.

In [ ]:
rewarding = []
for s in range(n_states):
    for a in range(n_actions):
        for prob, ns, r, term in P[s][a]:
            if r > 0:
                rewarding.append((s, ACTIONS[a], ns, prob))   # collect every reward-bearing transition

print("Every transition that yields reward +1:")
for s, a, ns, prob in rewarding:
    print(f"   from state {s} {divmod(s,4)} doing '{a}',  p={prob:.3f} -> goal state {ns}")
print("\nHoles (terminal, no reward):",
      [s for s in range(n_states) if env.unwrapped.desc.flatten()[s] == b'H'])

## The MDP is now fully specified

We have every piece of $(S, A, P, R, \gamma)$ written down explicitly:

- $S$: states 0&ndash;15, $A$: actions 0&ndash;3,
- $P$: the ⅓/⅓/⅓ slip model in `env.unwrapped.P`,
- $R$: +1 into state 15, else 0,
- $\gamma = 0.99$.

Because we **know** $P$ and $R$, we do not need to *learn* by trial and error &mdash; we can
**plan**: compute the optimal value function and policy directly. The tool for that is the
**Bellman optimality equation**.

## Solving the MDP by planning: value iteration

The optimal action-value of a state is the best expected return achievable from it. The
**Bellman optimality equation** writes it recursively &mdash; the value of a state is the
best, over actions, of the *expected* immediate reward plus the discounted value of where
you land:

$$V^*(s) = \max_{a} \sum_{s'} P(s' \mid s, a)\Big[\, R(s,a,s') + \gamma\, V^*(s') \,\Big]$$

The sum over $s'$ is exactly the slip distribution we just read out of `P`. **Value
iteration** turns this equation into an algorithm: start with $V = 0$, apply the right-hand
side as an update over and over, and $V$ converges to $V^*$. Then the optimal policy is the
action that achieves the max in each state.

In [ ]:
def one_step_lookahead(V, s, gamma):
    """Return the 4 action-values at state s: expected (reward + gamma*V[next]) under the slip model."""
    q = np.zeros(n_actions)
    for a in range(n_actions):
        # Average over every possible slip outcome of this action, weighted by its probability.
        q[a] = sum(prob * (r + gamma * V[ns] * (not term)) for prob, ns, r, term in P[s][a])
    return q                                   # (not term) zeroes the future for terminal landings

def value_iteration(gamma=0.99, theta=1e-9):
    V = np.zeros(n_states)
    while True:
        delta = 0.0                            # largest value change this sweep; used to test convergence
        for s in range(n_states):
            best = one_step_lookahead(V, s, gamma).max()   # Bellman optimality backup
            delta = max(delta, abs(best - V[s]))
            V[s] = best
        if delta < theta:                      # stop once updates are negligibly small
            break
    # Derive the greedy policy: the maximising action in each state.
    policy = np.array([one_step_lookahead(V, s, gamma).argmax() for s in range(n_states)])
    return V, policy

V_star, pi_star = value_iteration()
print("Optimal state values V* (4x4):")
print(np.round(V_star.reshape(4, 4), 3))
print("\nOptimal policy (arrows):")
arrows = np.array(["<", "v", ">", "^"])        # index 0..3 -> Left, Down, Right, Up
grid = arrows[pi_star].reshape(4, 4)
desc = env.unwrapped.desc.astype(str)
for r in range(4):
    print("   " + " ".join(g if desc[r][c] not in ("H","G") else desc[r][c] for c,g in enumerate(grid[r])))

## Visualise the solution

`plot_values` draws $V^*$ as a heat map (brighter = more valuable), and `plot_policy` draws
the optimal action in each cell as an arrow. We pass the action-values so the arrows reflect
the greedy choice.

In [ ]:
# Build the full Q* table (state x action) from V*, so plot_policy can draw the greedy arrows.
Q_star = np.vstack([one_step_lookahead(V_star, s, 0.99) for s in range(n_states)])

plot_values(V_star, env=env, title="V*(s): optimal state values on slippery ice")
plot_policy(Q_star, env=env, action_meanings={0:'L', 1:'D', 2:'R', 3:'U'})

## Why does the optimal policy look "wrong"?

Look at the arrows: in several cells the optimal move points **away** from the goal, even
into a wall. On firm ice that would be absurd. On slippery ice it is the smart play, and the
transition model explains why.

Take the start cell. If you intend **Right** (toward the goal), the slip model gives ⅓ Right,
⅓ **Up**, ⅓ **Down** &mdash; and Down from the start heads toward a hole. If instead you
intend **Left** (into the wall), the outcomes are ⅓ Left (bumps the wall, stay), ⅓ Up (wall,
stay), ⅓ Down. You *deliberately press against the wall* to remove the dangerous slip
directions, trading progress for safety. The optimal policy is the one that **maximises the
probability of eventually reaching the goal**, not the one that points at it most directly.
This is the whole reason a stochastic MDP needs to be *solved* rather than eyeballed.

## Evaluate: how good is the optimal policy?

Reward per episode on Frozen Lake is 0 or 1, so the average return over many episodes **is**
the success rate. We compare the optimal slippery policy against the optimal firm-ice policy
to size the cost of the ice.

In [ ]:
def policy_success(environment, policy, episodes=5000):
    """Run the deterministic 'policy' for many episodes; return the fraction that reach the goal."""
    return evaluate_policy(environment, lambda s: int(policy[s]), episodes)

# Optimal policy for the firm-ice twin, solved the same way, for a fair comparison.
def value_iteration_for(Pmodel, gamma=0.99, theta=1e-9):
    V = np.zeros(n_states)
    while True:
        delta = 0.0
        for s in range(n_states):
            q = [sum(p*(r+gamma*V[ns]*(not term)) for p,ns,r,term in Pmodel[s][a]) for a in range(n_actions)]
            best = max(q); delta = max(delta, abs(best-V[s])); V[s] = best
        if delta < theta: break
    return np.array([int(np.argmax([sum(p*(r+gamma*V[ns]*(not term)) for p,ns,r,term in Pmodel[s][a]) for a in range(n_actions)])) for s in range(n_states)])

pi_firm = value_iteration_for(P_firm)

print("Success rate of the OPTIMAL policy:")
print(f"   slippery ice     : {policy_success(env, pi_star):.3f}")
print(f"   firm ice         : {policy_success(env_firm, pi_firm):.3f}")
print("\nEven playing perfectly, the ice caps success well below 1.0 -- that gap is irreducible.")

## Watch the optimal policy act (optional, needs rendering)

This rolls out the optimal slippery policy and renders it. Expect the agent to sometimes
still slip into a hole &mdash; that is the environment's randomness, not a mistake in the
policy. Over many episodes it reaches the goal about three-quarters of the time, which is the
best any policy can do here.

In [ ]:
env_render = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")
test_agent(env_render, lambda s: int(pi_star[s]), episodes=5)

## Summary

- An MDP is $(S, A, P, R, \gamma)$. Making the ice slippery changes **only $P$**, the
  transition model &mdash; but that is enough to turn a trivial problem into a real one.
- On slippery Frozen Lake, an action goes as intended with probability **1/3** and slips to
  each perpendicular direction with probability **1/3**. Gymnasium exposes this exactly in
  `env.unwrapped.P[s][a]` as `(probability, next_state, reward, terminated)` tuples.
- The reward is **sparse**: +1 only on entering the goal; holes and goal are terminal.
- Because we know $P$ and $R$, we can **plan** with the **Bellman optimality equation** and
  **value iteration**, obtaining $V^*$ and the optimal policy without any trial and error.
- The optimal slippery policy is **counterintuitive** &mdash; it hugs walls to delete
  dangerous slip directions &mdash; because it maximises the *probability* of reaching the
  goal, not directness. Even so, the ice caps success near **0.74**.

**Next step:** in the real RL setting the agent does **not** get to read `env.unwrapped.P`.
It must *estimate* these dynamics implicitly from experience &mdash; which is exactly what
SARSA and Q-learning do in the other Frozen Lake notebooks.

## Resources

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2nd ed.), Ch. 3 (MDPs) and Ch. 4 (Dynamic Programming).
- Gymnasium Frozen Lake docs: https://gymnasium.farama.org/environments/toy_text/frozen_lake/